# BPE- ja WordPiece-tokenizerien testaus

Tässä notebookissa testataan kahta alisanoihin perustuvaa tokenisointimenetelmää:

- **BPE (Byte Pair Encoding)**
- **WordPiece**

Testidatana käytetään *Liisa ihmemaassa* -kirjan ensimmäistä kappaletta lyhennettynä noin 50 sanaan sekä englanniksi että suomeksi. Tekstit ovat public domain -aineistosta, ja notebook on tehty niin, että se toimii ilman ulkoisia tekstilatauksia.

In [1]:
# Tarvittavat kirjastot
# Jos tokenizers puuttuu ympäristöstä, asennetaan se.
import sys
import subprocess

try:
    import tokenizers
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tokenizers"])

from tokenizers import Tokenizer
from tokenizers.models import BPE, WordPiece
from tokenizers.trainers import BpeTrainer, WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Sequence, NFKC, Lowercase
from tokenizers.processors import TemplateProcessing
import pandas as pd

print("tokenizers version:", tokenizers.__version__)

tokenizers version: 0.23.1


## Testidata

Englanninkielinen teksti on *Alice's Adventures in Wonderland* -kirjan alusta. Suomenkielinen teksti on vastaava kohta vapaasti suomennettuna. Molemmat ovat lyhyitä, jotta tokenisaation tuloksia on helppo vertailla.

In [2]:
english_text = """
Alice was beginning to get very tired of sitting by her sister on the bank,
and of having nothing to do: once or twice she had peeped into the book her
sister was reading, but it had no pictures or conversations in it.
""".strip()

finnish_text = """
Liisa alkoi olla hyvin väsynyt istuessaan sisarensa vieressä joen rannalla,
eikä hänellä ollut mitään tekemistä: kerran tai kahdesti hän oli kurkistanut
kirjaan, jota sisar luki, mutta siinä ei ollut kuvia eikä keskusteluja.
""".strip()

texts = [english_text, finnish_text]

print("English text:", english_text)
print("Finnish text:", finnish_text)
print("English word count:", len(english_text.split()))
print("Finnish word count:", len(finnish_text.split()))

English text: Alice was beginning to get very tired of sitting by her sister on the bank,
and of having nothing to do: once or twice she had peeped into the book her
sister was reading, but it had no pictures or conversations in it.
Finnish text: Liisa alkoi olla hyvin väsynyt istuessaan sisarensa vieressä joen rannalla,
eikä hänellä ollut mitään tekemistä: kerran tai kahdesti hän oli kurkistanut
kirjaan, jota sisar luki, mutta siinä ei ollut kuvia eikä keskusteluja.
English word count: 43
Finnish word count: 32


## Tokenizerien luonti

Molemmat tokenizerit koulutetaan samalla pienellä tekstiaineistolla, jotta vertailu on selkeä. Käytetään Hugging Face `tokenizers` -kirjastoa.

Koska aineisto on hyvin pieni, tuloksia ei pidä ajatella tuotantotason tokenizerina. Tavoitteena on nähdä, miten BPE ja WordPiece pilkkovat sanoja eri tavoin.

In [3]:
def build_bpe_tokenizer(training_texts, vocab_size=80):
    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.normalizer = Sequence([NFKC(), Lowercase()])
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
    )
    tokenizer.train_from_iterator(training_texts, trainer=trainer)
    return tokenizer


def build_wordpiece_tokenizer(training_texts, vocab_size=80):
    tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
    tokenizer.normalizer = Sequence([NFKC(), Lowercase()])
    tokenizer.pre_tokenizer = Whitespace()
    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"],
        continuing_subword_prefix="##"
    )
    tokenizer.train_from_iterator(training_texts, trainer=trainer)
    return tokenizer

bpe_tokenizer = build_bpe_tokenizer(texts)
wordpiece_tokenizer = build_wordpiece_tokenizer(texts)

print("BPE vocab size:", bpe_tokenizer.get_vocab_size())
print("WordPiece vocab size:", wordpiece_tokenizer.get_vocab_size())

BPE vocab size: 80
WordPiece vocab size: 80


## Tokenisoinnin testaus englanniksi ja suomeksi

In [4]:
def tokenize_and_show(tokenizer, text, tokenizer_name, language):
    encoded = tokenizer.encode(text)
    return {
        "tokenizer": tokenizer_name,
        "language": language,
        "token_count": len(encoded.tokens),
        "tokens": encoded.tokens
    }

results = [
    tokenize_and_show(bpe_tokenizer, english_text, "BPE", "English"),
    tokenize_and_show(wordpiece_tokenizer, english_text, "WordPiece", "English"),
    tokenize_and_show(bpe_tokenizer, finnish_text, "BPE", "Finnish"),
    tokenize_and_show(wordpiece_tokenizer, finnish_text, "WordPiece", "Finnish"),
]

summary_df = pd.DataFrame([
    {"tokenizer": r["tokenizer"], "language": r["language"], "token_count": r["token_count"]}
    for r in results
])
summary_df

,tokenizer,language,token_count
0,BPE,English,102
1,WordPiece,English,131
2,BPE,Finnish,113
3,WordPiece,Finnish,146


In [5]:
for r in results:
    print("=" * 80)
    print(f"{r['tokenizer']} / {r['language']} / tokens: {r['token_count']}")
    print(r["tokens"])

BPE / English / tokens: 102
['a', 'li', 'ce', 'was', 'b', 'e', 'g', 'in', 'n', 'ing', 'to', 'g', 'e', 't', 'ver', 'y', 'ti', 're', 'd', 'of', 's', 'it', 't', 'ing', 'b', 'y', 'her', 'sist', 'er', 'on', 'the', 'b', 'an', 'k', ',', 'an', 'd', 'of', 'h', 'a', 'v', 'ing', 'no', 't', 'h', 'ing', 'to', 'd', 'o', ':', 'on', 'ce', 'or', 't', 'w', 'i', 'ce', 's', 'he', 'had', 'pe', 'e', 'pe', 'd', 'in', 'to', 'the', 'b', 'o', 'o', 'k', 'her', 'sist', 'er', 'was', 're', 'ad', 'ing', ',', 'b', 'ut', 'it', 'had', 'no', 'p', 'i', 'c', 't', 'u', 'r', 'es', 'or', 'c', 'on', 'ver', 'sa', 'ti', 'on', 's', 'in', 'it', '.']
WordPiece / English / tokens: 131
['al', '##i', '##ce', 'w', '##a', '##s', 'b', '##e', '##g', '##in', '##n', '##ing', 't', '##o', 'g', '##e', '##t', 'v', '##er', '##y', 't', '##i', '##r', '##e', '##d', 'o', '##f', 's', '##i', '##t', '##t', '##ing', 'b', '##y', 'her', 'sis', '##t', '##er', 'on', 't', '##he', 'b', '##an', '##k', ',', 'a', '##n', '##d', 'o', '##f', 'ha', '##v', '##ing', 

## Yksittäisten sanojen vertailu

Alla testataan sanoja, joissa on englannin ja suomen kannalta kiinnostavia piirteitä. Suomen sanat ovat usein pidempiä ja sisältävät taivutuspäätteitä, jolloin alisanoihin pilkkominen näkyy selvästi.

In [6]:
test_words = [
    "conversations",
    "beginning",
    "sister",
    "keskusteluja",
    "istuessaan",
    "sisarensa",
    "kurkistanut",
]

rows = []
for word in test_words:
    rows.append({
        "word": word,
        "BPE tokens": bpe_tokenizer.encode(word).tokens,
        "WordPiece tokens": wordpiece_tokenizer.encode(word).tokens,
    })

pd.DataFrame(rows)

,word,BPE tokens,WordPiece tokens
0,conversations,"[c, on, ver, sa, ti, on, s]","[c, ##o, ##n, ##v, ##er, ##sa, ##t, ##i, ##o, ..."
1,beginning,"[b, e, g, in, n, ing]","[b, ##e, ##g, ##in, ##n, ##ing]"
2,sister,"[sist, er]","[sis, ##t, ##er]"
3,keskusteluja,"[k, es, ku, s, te, lu, ja]","[k, ##es, ##k, ##u, ##s, ##t, ##e, ##l, ##u, #..."
4,istuessaan,"[ist, u, es, sa, an]","[i, ##s, ##t, ##u, ##es, ##sa, ##an]"
5,sisarensa,"[sisa, re, n, sa]","[sis, ##a, ##r, ##e, ##n, ##sa]"
6,kurkistanut,"[ku, r, k, ist, an, ut]","[ku, ##r, ##k, ##is, ##t, ##an, ##ut]"


## Pohdinta: BPE vs WordPiece

BPE ja WordPiece ovat molemmat alisanoihin perustuvia tokenisointimenetelmiä, mutta niiden merkintätapa ja tokenien muodostus eroavat hieman.

Tässä testissä **BPE** pilkkoo sanoja usein lyhyisiin merkkijonoihin ja pyrkii muodostamaan yleisiä merkkipareja tokeniksi. Koska aineisto on pieni, BPE ei opi kovin laajaa sanastoa, joten osa pidemmistä sanoista pilkkoutuu useiksi pieniksi osiksi.

**WordPiece** käyttää tässä `##`-merkintää jatko-osille. Tämä tekee näkyväksi sen, mitkä tokenit ovat sanan alussa ja mitkä ovat sanan jatkoa. Esimerkiksi pitkä sana voi näkyä muodossa `san`, `##an`, `##loppu`. Tämä on selkeä ero BPE-tokenisointiin, jossa jatko-osia ei merkitä samalla tavalla.

Käytännössä molemmat menetelmät auttavat käsittelemään sanoja, joita ei löydy kokonaisina sanastosta. Tämän takia ne ovat hyödyllisiä neuroverkoissa: mallin ei tarvitse tuntea jokaista sanaa kokonaisena, vaan se voi käsitellä sanan osina.

## Pohdinta: kielten väliset erot

Englanninkielinen teksti tokenisoituu yleensä hieman yksinkertaisemmin, koska sanat ovat tässä esimerkissä melko lyhyitä ja monia yleisiä sanoja esiintyy sellaisenaan. Esimerkiksi sanat kuten `alice`, `sister` ja `book` voivat säilyä kokonaisina tai lähes kokonaisina tokenina.

Suomenkielisessä tekstissä sanat ovat pidempiä ja sisältävät enemmän taivutusta. Esimerkiksi `istuessaan`, `sisarensa`, `keskusteluja` ja `kurkistanut` sisältävät vartalon lisäksi päätteitä tai johdoksia. Tällaiset sanat pilkkoutuvat helpommin useiksi alisanoiksi.

Tämä havainnollistaa, miksi suomen kaltaisissa kielissä tokenimäärä voi kasvaa verrattuna englantiin. Morfologisesti rikkaassa kielessä sama perusmerkitys voi esiintyä monessa taivutusmuodossa, jolloin alisanatokenisointi on erityisen tärkeää.

## Yhteenveto

Notebookissa testattiin kahta tokenizeria samalla englannin- ja suomenkielisellä tekstillä.

- BPE toimii yhdistämällä usein esiintyviä merkkipareja.
- WordPiece toimii samankaltaisesti, mutta merkitsee sanan jatkopalat `##`-etuliitteellä.
- Suomenkielinen teksti tuottaa usein enemmän alisanoja pitkien ja taipuneiden sanojen takia.
- Englanninkielinen teksti säilyttää tässä testissä useammin lyhyitä sanoja kokonaisina.